### 🔎 Leitura, Conversão do Arquivo JSON para CSV e formatação

#### Importação do Arquivo do Bucket

In [10]:
from    dotenv import load_dotenv
import  os
import  pandas as pd

# Carrega variáveis do .env
if not os.path.exists("../.env"):
    print("Arquivo .env não encontrado.")
else:
    load_dotenv("../.env", override=True)
    print("ENV carregado")

# Variáveis de ambiente
access_key= os.getenv("AWS_ACCESS_KEY_ID")
secret_key = os.getenv("AWS_SECRET_ACCESS_KEY")
session_token = os.getenv("AWS_SESSION_TOKEN")
region = os.getenv("AWS_DEFAULT_REGION")
url_dados = os.getenv("URL_S3_BUCKET")

try:
    df = pd.read_csv(
        url_dados,
        storage_options={
            "key": access_key,
            "secret": secret_key,
            "token": session_token
        }
    )
    print("Arquivo carregado com sucesso!")
except Exception as e:
    print(f"Erro ao carregar dados: {e}")

ENV carregado
Arquivo carregado com sucesso!


#### Formatando a Tabela
Retirando valores nulos, convertendo colunas para tipos corretos e valores extensos da coluna 'Valor' abreviados

In [2]:
# Retirando valores nulos e linhas sem valor
df = df.replace("...", None)
df = df.dropna(subset=["Valor"])

# Convertendo as colunas para os tipos corretos
df["Valor"] = (pd.to_numeric(df["Valor"], errors="coerce")*1000)    # Função de Conversão para valores reais
df["Valor"] = df["Valor"].astype(float)             

# Formatação da coluna "Ano" para datetime
df["Ano"] = pd.to_datetime(df["Ano"], format="%Y")                  
df["Ano"] = df["Ano"].dt.year                                       


# Apenas pra visualização bonita, float continua intacto
def formatar_valor(x):
    if x >= 1e12: return f'{x/1e12:,.2f} trilhões'.replace('.', ',')
    if x >= 1e9:  return f'{x/1e9:,.2f} bilhões'.replace('.', ',')
    if x >= 1e6:  return f'{x/1e6:,.2f} milhões'.replace('.', ',')
    return f'{x:,.2f}'.replace('.', ',')

# Exportar DataFrame para CSV
def exportar_csv(df, nome_arquivo):
    df.to_csv(nome_arquivo, index=False)
    print(f"DataFrame exportado para {nome_arquivo}")

## 📊 Análises baseadas no PIB

- **PIB Total, PIB Médio Nacional e PIB per capita Médio**  
- **Top 10 Municípios com mais PIB**  
- **Ranking dos 27 estados no PIB**  
- **Ranking das regiões**  
- **10 Municípios da Região Sudeste com menos de 10 milhões**

### PIB Total e Média Nacional

In [ ]:
# Cálculo do PIB Total Nacional
pib_brasil = df["Valor"].sum()
print(f"Output - PIB Total Nacional: {formatar_valor(pib_brasil)}")

# Cálculo do PIB Médio Nacional
pib_medio = formatar_valor(df["Valor"].mean())
print(f"Output - PIB Médio Nacional: {pib_medio}")

# Quantidade de municípios
qtd_municipios = df["Município"].nunique()

df_resumo = pd.DataFrame({
    "Métrica": ["PIB Total Nacional", "Média do PIB", "Quantidade de Municípios"],
    "Valor": [
        formatar_valor(pib_brasil),
        pib_medio,
        qtd_municipios
    ]
})

Output - PIB Total Nacional: 10,94 trilhões
Output - PIB Médio Nacional: 1,96 bilhões


### Top 10 Municípios com maior PIB em 2023

De acordo com a base de dados do IBGE e considerado como critério mais utilizado por economistas é o PIB per Capita para definir um município como PIB Alto.
Sendo assim, acima de R$100.000,00 são considerados PIB Alto. Aqueles acima de R$200.000,00 possuem atividades industriais pesadas, extração de petróleo ou grande força no agronegócio.

Válido saber que apenas 10 municípios (liderados por São Paulo, Rio de Janeiro e Brasília) concentram cerca de 25% de todo o PIB Nacional. 

O PIB per capita médio nacional foi de R$51.693,92.

In [4]:
# Classificando os municípios com base no valor do PIB
df["Classe PIB"] = df["Valor"].apply( lambda x: "Alto PIB" if x > 20000000000 else "PIB Médio/Baixo") # PIB Alto = acima de 20 bilhões

# Calculando % de cada município em relação ao PIB total e formatando como porcentagem
percentual_por_municipio = ((df["Valor"] / pib_brasil) * 100).round(2)
df["% do PIB Total"] = percentual_por_municipio

df_municipio = df[["Município", "Valor", "Classe PIB", "% do PIB Total"]]

df_municipio = df_municipio.sort_values("% do PIB Total", ascending=False).head(30)

# Formatação dos valores
df_municipio["Valor"] = df_municipio["Valor"].apply(formatar_valor)
df_municipio["% do PIB Total"] = df_municipio["% do PIB Total"].map(lambda x: f"{x:.2f}%")

### Ranking dos 27 Estados por PIB

In [5]:
# Extrair estados
df["Estado"] = df["Município"].str.split(" - ").str[1]

# PIB por estado
df_estado = df.groupby("Estado", as_index=False)["Valor"].sum()

# Total Brasil
total_br = df_estado["Valor"].sum()

# Percentual
df_estado["% do PIB Total"] = ((df_estado["Valor"] / total_br) * 100).round(2)

# Quantidade de municípios
df_estado["Quantidade de Municípios"] = df.groupby("Estado")["Município"].nunique().values

# Ordenar
df_estado = df_estado.sort_values("% do PIB Total", ascending=False)

# Formatação
df_estado["Valor"] = df_estado["Valor"].apply(formatar_valor)
df_estado["% do PIB Total"] = df_estado["% do PIB Total"].map(lambda x: f"{x:.2f}%")


### Ranking Regiões com maior PIB
Norte, Nordeste, Centro-Oeste, Sudeste e Sul

In [7]:
# Mapear regiões
sul = ['PR', 'SC', 'RS']
sudeste = ['SP', 'RJ', 'MG', 'ES']
nordeste = ['BA', 'PE', 'CE', 'RN', 'PB', 'AL', 'SE', 'MA', 'PI']
norte = ['AM', 'PA', 'AC', 'RO', 'RR', 'AP', 'TO']
centro_oeste = ['MT', 'MS', 'GO', 'DF']

def categorizar_regiao(estado):
    if estado in sul: return 'Sul'
    if estado in sudeste: return 'Sudeste'
    if estado in nordeste: return 'Nordeste'
    if estado in norte: return 'Norte'
    return 'Centro-Oeste'

df['Região'] = df['Estado'].apply(categorizar_regiao)

# Agrupar PIB por região
df_valor_regiao = df.groupby("Região", as_index=False)["Valor"].sum()

# Total do Brasil
total_br = df["Valor"].sum()

# Percentual
df_valor_regiao["% do PIB Total"] = ((df_valor_regiao["Valor"] / total_br) * 100).map(lambda x: f"{x:.2f}%")

# Ordenar
df_valor_regiao = df_valor_regiao.sort_values("% do PIB Total", ascending=False)

df_valor_regiao["Valor"] = df_valor_regiao["Valor"].apply(formatar_valor)

### 10 Municípios da Região Sudeste com menos de 10 milhões de PIB

In [8]:
### Municípios com menos de 100 milhões de PIB no Estado de São Paulo

df_menor_pib = df[(df["Valor"] < 100000000) & (df["Região"] == "Sudeste")]

df_municipios_menor_pib = df_menor_pib[["Município", 
                                        "Estado", 
                                        "Valor"
                                        ]]
df_municipios_menor_pib["Valor"] = df_municipios_menor_pib["Valor"].apply(formatar_valor)

df_municipios_menor_pib = df_municipios_menor_pib.sort_values("Valor", ascending=True).head(10)


## 🟢 Exportar análises para o Bucket S3

In [9]:
# ---- Exportar localmente ----
# exportar_csv(df_resumo, "resumo_pib.csv")
# exportar_csv(df_municipio, "top_30_municipios.csv")
# exportar_csv(df_estado, "rank_pib_estados.csv")
# exportar_csv(df_valor_regiao, "rank_pib_regioes.csv")
# exportar_csv(df_municipios_menor_pib, "municipios_menor_pib_sudeste.csv")

# Exportar para o S3
path="s3://challenge-sprint6/analises"

df_resumo.to_csv(
    f"{path}/resumo_pib.csv",
    index=False,
    storage_options={
        "key": access_key,
        "secret": secret_key,
        "token": session_token
    }
)

df_municipio.to_csv(
    f"{path}/top_30_municipios.csv",
    index=False,
    storage_options={
                    "key": access_key,
                    "secret": secret_key,
                    "token": session_token
                    }
)

df_estado.to_csv(
    f"{path}/rank_pib_estados.csv",
    index=False,
    storage_options={
                    "key": access_key,
                    "secret": secret_key,
                    "token": session_token
                    }
)

df_valor_regiao.to_csv(
    f"{path}/rank_pib_regioes.csv",
    index=False,
    storage_options={
                    "key": access_key,
                    "secret": secret_key,
                    "token": session_token
                    }
)

df_municipios_menor_pib.to_csv(
    f"{path}/municipios_menor_pib_sudeste.csv",
    index=False,
    storage_options={
                    "key": access_key,
                    "secret": secret_key,
                    "token": session_token
                    }
)
